# Wilson-Cowan Higher-Order Model

---
## **1. Theoretical Introduction**

### **1.1. Wilson-Cowan Model with Higher-Order Interactions**
The Wilson-Cowan model with higher-order interactions extends the classic model by incorporating triadic interactions between excitatory populations.

#### **Model Equations:**
- **Excitatory Population:**
  $$
  \tau_E \frac{dE_i}{dt} = -E_i + S\left(c_{EE} E_i - c_{IE} I_i + P + K_3 \sum_{j,k} T_{ijk} E_j E_k\right)
  $$

- **Inhibitory Population:**
  $$
  \tau_I \frac{dI_i}{dt} = -I_i + S\left(c_{EI} E_i - c_{II} I_i + Q\right)
  $$

#### **Sigmoid Function:**
$$
S(v) = \frac{1}{1 + e^{-a v}}
$$

#### **Higher-Order Interaction Tensor:**
$$
I_i^{net} = K_3 \sum_{j,k} T_{ijk} E_j E_k
$$

---
## **2. Initial Setup**
We import the functions already defined in the `Scripts` folder.


In [4]:
import sys
from tqdm import tqdm
sys.path.append('../')  # Ensure Python can find the 'Scripts' folder

#from Scripts.parameters import *
#from Scripts.parameters_random import *
from Scripts.parameters import *
from Scripts.dynamics import *
from Scripts.simulation import *
from Scripts.oscillation_detection import *
from Scripts.metrics import *
from Scripts.visualization import *

print("Functions loaded successfully.")

# Parameter ranges for oscillation detection
P_values = np.linspace(1.0, 10.0, 20)
K_values = np.linspace(0.0, 1.0, 20)
K3_values = np.linspace(0.0, 1.0, 20)

Functions loaded successfully.


---
## **4. Oscillation Detection in $(P \times K_3)$ Space**
We perform a parameter sweep to detect oscillations in the $(P \times K_3)$ space.


In [6]:
P_values = [3,4,5,6]
K3_values = np.linspace(0.0, 1.0, 20)

# Matrix to store oscillation detection
oscillation_map = np.zeros((len(P_values), len(K3_values)), dtype=int)

T=10
# Matrices to store metrics with noise
TC_map = np.zeros((len(P_values), len(K3_values)))
DTC_map = np.zeros((len(P_values), len(K3_values)))
Cumulant_map = np.zeros((len(P_values), len(K3_values)))
PowerCorr_map = np.zeros((len(P_values), len(K3_values)))
Entropy_map = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica
Entropy_map_skew = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica
Entropy_map_kurt = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica

#initial state random fixed
seed = 42
rng = np.random.default_rng(seed)
state0 = 0.1 * rng.standard_normal(2 * N_nodes)



for i, P_val in tqdm(enumerate(P_values)):
    for j, K3_val in enumerate(K3_values):
        # Simulation using the existing function for higher-order interactions
        P_vec = np.array([P_val, P_val, P_val])
        t, states = simulate_wc_higher_order_additive(state0, P_vec, K3_val, T_ho, T, dt)
       
        #deleting the firts 20% of the signal
        start_idx = int(0.2 * len(t))

        E_node = states[start_idx:, :N_nodes]
        I_node = states[start_idx:, N_nodes:]

        # Oscillation detection using the existing function
        is_lc1, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc2, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc3, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        oscillation_map[i, j]  = int((is_lc1==True) and (is_lc2==True) and (is_lc3==True))

        #deleting the firts 30% of the signal
        E = states[start_idx:, :N_nodes]

        # Data matrix for higher-order metrics
        X = E.copy()

        # Covariance matrix
        Sigma = np.cov(X.T)

        # Higher-order metrics using the existing functions
        TC_map[i, j] = TC_gauss(Sigma)
        DTC_map[i, j] = DTC_gauss(Sigma)
        Entropy_map[i, j] = entropy_gauss(Sigma)  # Cálculo de la entropía
        skew, kurt = dev_gauss(X)  # Cálculo de la entropía
        Entropy_map_skew[i, j] = skew
        Entropy_map_kurt[i, j] = kurt
        if X.shape[1] == 3:
            Cumulant_map[i, j] = cumulants(X)
            PowerCorr_map[i, j] = powercorr(X)


    if (i + 1) % 5 == 0:
        print(f"Progress: {i+1}/{len(P_values)}")

# Visualization using the existing function
#plot_oscillation_map(oscillation_map, P_values, K3_values, title="Oscillation Detection (P-K3)")


#cleaing the matrix
TC_clean = clean_rowwise_signed(TC_map, n_std=2)
DTC_clean = clean_rowwise_signed(DTC_map, n_std=2)
Cumulant_clean = clean_rowwise_signed(Cumulant_map,n_std=2)
PowerCorr_clean = clean_rowwise_signed(PowerCorr_map,n_std=2)
Skew_clean = clean_rowwise_signed(Entropy_map_skew,n_std=2)
Kurt_clean = clean_rowwise_signed(Entropy_map_kurt,n_std=2)

# visualization the matrix cleaned
# plot_metrics_contour(, P_values, K3_values, title="Oscilation detection")
plot_lines_with_area(oscillation_map, x_values=K3_values, title="Oscilation Detection")

plot_lines_with_area(TC_clean, x_values=K3_values, title="Total Correlation (TC) without Noise")
plot_lines_with_area(DTC_clean, x_values=K3_values, title="Dual Total Correlation (DTC) without Noise")
plot_lines_with_area(Cumulant_clean, x_values=K3_values, title="Third-Order Cumulant without Noise")
plot_lines_with_area(PowerCorr_clean, x_values=K3_values, title="Triple Power Correlation without Noise")
plot_lines_with_area(Skew_clean, x_values=K3_values, title="Skew with Noise")
plot_lines_with_area(Kurt_clean, x_values=K3_values, title="Kurt with Noise")

4it [00:40, 10.08s/it]


---
## **6. Model with Stochastic Noise and Higher-Order Interactions**
We add stochastic noise to the excitatory population (E) using the Euler-Maruyama method with higher-order interactions.

To include biological noise, we extend the excitatory population dynamics using the **Euler-Maruyama method**:

$$
E_i(t + \Delta t) = E_i(t) + \frac{\Delta t}{\tau_E} \left(-E_i(t) + S\left(c_{EE} E_i - c_{IE} I_i + P + K_3 \sum_{j,k} T_{ijk} E_j E_k\right)\right) + \frac{\sigma_E \sqrt{\Delta t}}{\tau_E} \xi_i
$$

- $(\sigma_E)$: Noise intensity for the excitatory population.
- $(\xi_i \sim \mathcal{N}(0, 1))$: Gaussian white noise.

---
## **7. Oscillation Detection in $(P \times K$) Space** with noise
We perform a parameter sweep to detect oscillations in the $(P \times K$) space.


In [7]:
P_values = [3,4,5,6]
K3_values = np.linspace(0.0, 1.0, 20)
sigma_E = 0.03
T=450


# Matrices to store metrics with noise
TC_map_noise = np.zeros((len(P_values), len(K_values)))
DTC_map_noise = np.zeros((len(P_values), len(K_values)))
Cumulant_map_noise = np.zeros((len(P_values), len(K_values)))
PowerCorr_map_noise = np.zeros((len(P_values), len(K_values)))
Entropy_map_noise = np.zeros((len(P_values), len(K_values)))  # Nueva métrica
Entropy_map_skew = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica
Entropy_map_kurt = np.zeros((len(P_values), len(K3_values)))  # Nueva métrica



# Matrix to store oscillation detection with noise
oscillation_map_noise = np.zeros((len(P_values), len(K3_values)), dtype=int)
# mean_map = np.zeros((len(P_values), len(K_values), N_nodes))

#initial state
seed = 42
rng = np.random.default_rng(seed)
state0 = 0.1 * rng.standard_normal(2 * N_nodes)

for i, P_val in tqdm(enumerate(P_values)):
    for j, K3_val in enumerate(K3_values):
        # Simulation with noise using the existing function for higher-order interactions
        # t, states = simulate_wc_stochastic(state0, P_val, K3_val, M, T, dt, sigma_E=sigma_E, higher_order=True, T_ho=T_ho)
        P_vec = np.array([P_val, P_val, P_val])
        t, states = simulate_wc_stochastic_additive(state0, P_vec, K3_val, M, T, dt, sigma_E = sigma_E, higher_order=True, T_ho=T_ho)

        start_idx = int(0.2 * len(t))

        E_node = states[start_idx:, :N_nodes]
        I_node = states[start_idx:, N_nodes:]

        # Oscillation detection using the existing function
        is_lc1, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc2, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        is_lc3, _, _, _ = detect_limit_cycle_poincare(E_node[:, 0], I_node[:, 0], dt, threshold_ratio=0.05, period_cv_threshold=0.05)
        oscillation_map_noise[i, j]  = int((is_lc1==True) and (is_lc2==True) and (is_lc3==True))

        # quitar transiente (ej. mitad inicial)
        # cut = int(0.5 * len(t))
        # E_ss = E_node[cut:, :]
        # mean firing por nodo
        # mean_map[i, j, :] = np.mean(E_ss, axis=0)


        # Remove transient
        E = states[start_idx:, :N_nodes]

        # Data matrix for higher-order metrics
        X = E.copy()

        # Covariance matrix
        Sigma = np.cov(X.T)

        # Higher-order metrics using the existing functions
        TC_map_noise[i, j] = TC_gauss(Sigma)
        DTC_map_noise[i, j] = DTC_gauss(Sigma)
        skew, kurt = dev_gauss(X)  # Cálculo de la entropía
        Entropy_map_skew[i, j] = skew
        Entropy_map_kurt[i, j] = kurt
        #Entropy_map_noise[i, j] = dev_gauss(Sigma)  # Cálculo de la entropía
        if X.shape[1] == 3:
            Cumulant_map_noise[i, j] = cumulants(X)
            PowerCorr_map_noise[i, j] = powercorr(X)

    if (i + 1) % 5 == 0:
        print(f"Progress: {i+1}/{len(P_values)}")

# Visualization using the existing function
# plot_oscillation_map(oscillation_map_noise, P_values, K3_values, title="Oscillation Detection (P-K3) with Noise")

# Visualization using the existing functions

#cleaing the matrix
TC_clean = clean_rowwise_signed(TC_map_noise, n_std=2)
DTC_clean = clean_rowwise_signed(DTC_map_noise, n_std=2)
Cumulant_clean = clean_rowwise_signed(Cumulant_map_noise,n_std=2)
PowerCorr_clean = clean_rowwise_signed(PowerCorr_map_noise,n_std=2)
Skew_clean = clean_rowwise_signed(Entropy_map_skew,n_std=2)
Kurt_clean = clean_rowwise_signed(Entropy_map_kurt,n_std=2)

# visualization the matrix cleaned
plot_lines_with_area(oscillation_map_noise, x_values=K3_values, title="Oscilation Detection")

plot_lines_with_area(TC_clean, x_values=K3_values, title="Total Correlation (TC) without Noise")
plot_lines_with_area(DTC_clean, x_values=K3_values, title="Dual Total Correlation (DTC) without Noise")
plot_lines_with_area(Cumulant_clean, x_values=K3_values, title="Third-Order Cumulant without Noise")
plot_lines_with_area(PowerCorr_clean, x_values=K3_values, title="Triple Power Correlation without Noise")
plot_lines_with_area(Skew_clean, x_values=K3_values, title="Skew with Noise")
plot_lines_with_area(Kurt_clean, x_values=K3_values, title="Kurt with Noise")

4it [09:16, 139.19s/it]
